这是**遗传算法 (Genetic Algorithm, GA)** 的详细解析。

遗传算法是由 John Holland 于 20 世纪 70 年代受达尔文进化论启发而提出的**启发式全局搜索算法**。它模拟了自然界中“物竞天择，适者生存”的过程，通过**选择、交叉和变异**算子，在复杂的搜索空间中寻找最优解。

---

### 一、 数学原理与深度推导

**核心思想**：将问题的解编码为“染色体”（通常是二进制字符串或实数向量），通过不断的演化，使得种群的整体“适应度”（目标函数值）不断提高。

#### 1. 编码与映射（数学表示）
假设我们要优化函数 $f(x)$，首先需要将决策变量 $x$ 映射到基因空间。
*   **二进制编码**：若 $x \in [L, U]$，使用 $k$ 位二进制数表示，则精度为 $\Delta = \frac{U - L}{2^k - 1}$。
*   **映射公式**：$x = L + (\text{decimal value of binary string}) \cdot \Delta$。

#### 2. 模式定理 (Schema Theorem) —— 算法之所以有效的数学证明
这是 GA 的数学基石。**模式 (Schema)** 是指在染色体某些位置上具有特定基因值的位串（用 $\{0, 1, *\}$ 表示，其中 $*$ 是通配符）。

**推导过程：**
设 $m(H, t)$ 为 $t$ 代中匹配模式 $H$ 的个体数量。
*   **选择算子影响**：采用轮盘赌选择，个体被选中的概率 $P_i = \frac{f_i}{\bar{f} \cdot n}$。则下一代中模式 $H$ 的预期数量为 $m(H, t) \cdot \frac{f(H, t)}{\bar{f}}$。
*   **交叉算子破坏概率**：设模式 $H$ 的定义长度为 $\delta(H)$（首尾固定位距离），染色体长度为 $L$，交叉概率为 $P_c$。模式被破坏的概率约等于 $P_c \cdot \frac{\delta(H)}{L-1}$。
*   **变异算子破坏概率**：设模式中固定位数为 $o(H)$，变异概率为 $P_m$。模式存活概率约为 $(1-P_m)^{o(H)} \approx 1 - o(H)P_m$。

**结论（模式定理公式）：**
$$ E[m(H, t+1)] \ge m(H, t) \cdot \frac{f(H)}{\bar{f}} \left[ 1 - P_c \frac{\delta(H)}{L-1} - o(H)P_m \right] $$
**数学意义**：具有**短定义长度、低阶、且适应度高于群体平均水平**的模式（即“积木块”），其样本数在种群中呈指数级增长。这就是著名的**积木块假设 (Building Block Hypothesis)**。

---

### 二、 算法流程

1.  **初始化**：随机生成 $N$ 个个体组成的初始种群。
2.  **个体评价**：计算每个个体的适应度 $F_i$（通常直接取目标函数值，或对其进行正向平移）。
3.  **选择 (Selection)**：按照“适者生存”原则挑选优秀个体。常用**轮盘赌法 (Roulette Wheel Selection)**。
4.  **交叉 (Crossover)**：模拟有性生殖，交换两个个体的部分基因，产生新性状。
5.  **变异 (Mutation)**：以极小概率随机改变某个基因位，维持种群多样性，防止**早熟（陷入局部最优）**。
6.  **进化**：重复上述过程，直到满足终止条件。

---

### 三、 适用场景
1.  **复杂多峰函数优化**：函数有大量局部极值，梯度下降法易失效。
2.  **组合优化问题**：如旅行商问题 (TSP)、装箱问题、排课问题。
3.  **参数搜索**：如神经网络权重优化、特征选择。
4.  **黑盒系统**：完全不知道函数解析式，只需输入 $x$ 得到输出 $y$。

---

### 四、 Python 代码框架

我们将求解一个高频震荡的函数 $f(x) = x \cdot \sin(10\pi \cdot x) + 2.0$ 在 $[ -1, 2]$ 上的最大值。

```python
import numpy as np
import matplotlib.pyplot as plt

# 1. 目标函数 (适应度函数)
def fitness_func(x):
    return x * np.sin(10 * np.pi * x) + 2.0

class GeneticAlgorithm:
    def __init__(self, bounds, pop_size=100, genome_len=20, n_generations=200, p_c=0.8, p_m=0.01):
        self.lb, self.ub = bounds
        self.pop_size = pop_size
        self.genome_len = genome_len
        self.n_generations = n_generations
        self.p_c = p_c # 交叉概率
        self.p_m = p_m # 变异概率
        
        # 初始化种群 (随机 0, 1 矩阵)
        self.pop = np.random.randint(0, 2, (pop_size, genome_len))

    def decode(self, pop):
        """将二进制编码转换为实数"""
        precision = (self.ub - self.lb) / (2**self.genome_len - 1)
        # 将矩阵的每一行从二进制转为十进制
        res = pop.dot(2**np.arange(self.genome_len)[::-1])
        return self.lb + res * precision

    def select(self, fitness):
        """轮盘赌选择"""
        # 确保适应度为正
        idx = np.random.choice(np.arange(self.pop_size), size=self.pop_size, replace=True,
                               p=fitness / fitness.sum())
        return self.pop[idx]

    def crossover(self, parent, pop):
        """单点交叉"""
        if np.random.rand() < self.p_c:
            i_ = np.random.randint(0, self.pop_size, size=1) # 选另一个
            cross_points = np.random.randint(0, 2, size=self.genome_len).astype(bool)
            parent[cross_points] = pop[i_, cross_points]
        return parent

    def mutate(self, child):
        """变异"""
        for i in range(self.genome_len):
            if np.random.rand() < self.p_m:
                child[i] = 1 if child[i] == 0 else 0
        return child

    def evolve(self):
        best_fitness_history = []
        
        for gen in range(self.n_generations):
            # 1. 解码并计算适应度
            x_values = self.decode(self.pop)
            fitness = fitness_func(x_values)
            
            # 记录本代最佳
            best_idx = np.argmax(fitness)
            best_fitness_history.append(fitness[best_idx])
            
            # 2. 选择
            self.pop = self.select(fitness)
            pop_copy = self.pop.copy()
            
            # 3. 交叉与变异
            for i in range(self.pop_size):
                child = self.crossover(self.pop[i], pop_copy)
                child = self.mutate(child)
                self.pop[i] = child
                
        return self.decode(self.pop), fitness, best_fitness_history

# ================= 运行示例 =================

if __name__ == "__main__":
    ga = GeneticAlgorithm(bounds=(-1, 2), n_generations=100)
    final_pop_x, final_fitness, history = ga.evolve()
    
    best_idx = np.argmax(final_fitness)
    print("-" * 30)
    print(f"全局最优解 x: {final_pop_x[best_idx]:.6f}")
    print(f"最大值 f(x): {final_fitness[best_idx]:.6f}")

    plt.plot(history)
    plt.title("Genetic Algorithm Convergence")
    plt.xlabel("Generation")
    plt.ylabel("Best Fitness")
    plt.grid(True)
    plt.show()
```

---

### 五、 深入数学推导：交叉与变异的博弈

在论文中，讨论参数 $P_c$ 和 $P_m$ 的选取是体现深度的关键：

1.  **交叉概率 $P_c$ 的作用**：
    *   $P_c$ 较大时，种群产生新个体的速度快，有利于探索未知的搜索空间。
    *   但若 $P_c$ 过大，高适应度的基因结构（积木块）容易被破坏，导致搜索变成“随机搜索”。

2.  **变异概率 $P_m$ 的作用**：
    *   $P_m$ 的本质是**引入噪声**，跳出局部最优。
    *   推导逻辑：由于选择和交叉会使种群趋向同质化（多样性丧失），$P_m$ 保证了搜索过程具有**遍历性 (Ergodicity)**，即理论上可以到达搜索空间任何一点。

---

### 六、 论文写作与“修修补补”建议

1.  **引入“精英保留策略” (Elitism)**：
    *   **技巧**：直接将每一代中最优秀的个体不经过交叉变异，直接复制到下一代。
    *   **论文表述**：为了防止进化过程中最优解因随机干扰而丢失，本文采用了精英保留机制，确保算法的收敛性质。

2.  **参数自适应 (Adaptive GA)**：
    *   **进阶推导**：让 $P_c$ 和 $P_m$ 随适应度变化。当个体适应度高时，降低变异率以保护优良基因；当种群趋于集中时，提高变异率以打破停滞。
    *   公式参考：$P_m = k \cdot \frac{f_{max} - f}{f_{max} - \bar{f}}$。

3.  **编码方式讨论**：
    *   如果处理的是连续变量，建议在论文中提到使用 **Gray Code (格雷码)**。
    *   **理由**：二进制编码中，3 (011) 到 4 (100) 的变换需要翻转 3 位基因，存在“汉明悬崖”。格雷码相邻数仅差 1 位，更有利于局部搜索。

4.  **结果展示**：
    *   一定要展示**种群均值**和**种群最大值**随代数变化的收敛图。若两条线逐渐合拢，说明算法收敛稳定。

**下一类算法，你想听哪一个的深度数学推导？**